# Evaluate — Full Pipeline Comparison

Compares crop-based pipelines and YOLO against CVAT ground truth.

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 (browse) → 12 → 13 → 14

**Output per run** — e.g. `evaluation/2025-05-21_14-30-00/`:

| File | Contents |
|---|---|
| `eval_{timestamp}.xlsx` | All metrics — 9 sheets |
| `report.json` | Printed output split by section: `overall_metrics`, `per_plot_metrics`, `combined_analysis`, `unique_detections` |
| `threshold_analysis.png` | PR curve chart |


## Cell 1 — Environment  ← edit your path here

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
from pathlib import Path
from datetime import datetime

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import zipfile, os
    DRIVE_ROOT  = Path('/content/drive/MyDrive')
    ZIP_PATH    = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO  = Path('/content/pollinator-colab')

    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} to /content/ ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('\u2713 Extracted')
    else:
        print('\u2713 Already extracted')

    BASE_DIR   = EXTRACT_TO
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    BASE_DIR   = Path('/Users/lianshi/Downloads/bachelor thesis'
                      '/automated-ecological-image-analysis'
                      '/ml-pipelines/notebooks/pollinator-classification')
    DRIVE_BASE = BASE_DIR

IMAGE_ROOT        = BASE_DIR  / 'Insects_images' / 'e2e_evaluation_images'
GT_ANN_ROOT       = BASE_DIR  / 'Insects_images' / 'e2e_yolo_annotations'
MODEL_DIR         = BASE_DIR  / 'models'
CROP_RESULTS_ROOT = (DRIVE_BASE if IN_COLAB else BASE_DIR) / 'Insects_images' / 'crop_results'
YOLO_RESULTS_ROOT = (DRIVE_BASE if IN_COLAB else BASE_DIR) / 'Insects_images' / 'yolo_results'

# ── Each run gets its own timestamped folder, old ones are never overwritten ─
RUN_TS   = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
EVAL_DIR = BASE_DIR / 'evaluation' / RUN_TS
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env              : {"Colab" if IN_COLAB else "Local"}')
print(f'Run timestamp    : {RUN_TS}')
print(f'EVAL_DIR         : {EVAL_DIR}')
print(f'IMAGE_ROOT       : {IMAGE_ROOT}  exists={IMAGE_ROOT.exists()}')
print(f'GT_ANN_ROOT      : {GT_ANN_ROOT}  exists={GT_ANN_ROOT.exists()}')
print(f'CROP_RESULTS_ROOT: {CROP_RESULTS_ROOT}  exists={CROP_RESULTS_ROOT.exists()}')
print(f'YOLO_RESULTS_ROOT: {YOLO_RESULTS_ROOT}  exists={YOLO_RESULTS_ROOT.exists()}')


## Cell 2 — Evaluation config  ← edit here

In [ ]:
# ── Which runs to evaluate ──────────────────────────────────────
CROP_RUNS = {
    'run_01': CROP_RESULTS_ROOT / 'run_02_large_motion_on_thr15_veg40-85_sat60_minarea200',
    # 'run_02_lm_off': CROP_RESULTS_ROOT / 'run_02_lm_off',

}
YOLO_RUNS = {
    'yolo_run_01': YOLO_RESULTS_ROOT / 'yolo_run_01',
}

GT_CLASSES        = ['bumblebee', 'fly', 'butterfly', 'other']
YOLO_GT_CLASSES   = ['fly', 'butterfly', 'other']
STRIP_HEIGHT = 120

print('Runs to evaluate:')
for name, path in {**CROP_RUNS, **YOLO_RUNS}.items():
    print(f'  {name}: exists={path.exists()}')


## Cell 3 — Imports

In [ ]:
import csv, json as _json, io, sys
from pathlib import Path
from collections import defaultdict
import numpy as np, cv2
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Per-section stdout capture for report.json ──────────────────────
class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, data):
        for s in self.streams: s.write(data)
    def flush(self):
        for s in self.streams: s.flush()

_report_sections = {}   # section_name -> captured text
_current_buf     = None
_real_stdout     = sys.stdout

def _start_capture(section):
    global _current_buf
    _current_buf = io.StringIO()
    sys.stdout   = _Tee(_real_stdout, _current_buf)

def _stop_capture(section):
    global _current_buf
    sys.stdout = _real_stdout
    if _current_buf is not None:
        _report_sections[section] = _current_buf.getvalue()
        _current_buf = None

print('✓ Imports done')


## Cell 4 — Load ground truth

Reads CVAT YOLO 1.1 annotations.
Uses **original image dimensions** for coordinate conversion (not stripped height).

In [ ]:
def load_gt(gt_ann_root, image_root, gt_classes, strip_height=120):
    print('Loading ground truth...')
    gt = {}
    for cam_dir in sorted(Path(gt_ann_root).iterdir()):
        if not cam_dir.is_dir(): continue
        lbl_dir = cam_dir / 'obj_train_data'
        img_dir = Path(image_root) / cam_dir.name
        if not lbl_dir.exists() or not img_dir.exists(): continue
        names_f = cam_dir / 'obj.names'
        cls_names = ([l.strip() for l in names_f.read_text().splitlines() if l.strip()]
                     if names_f.exists() else gt_classes)
        n_cam = 0
        for txt in sorted(lbl_dir.glob('*.txt')):
            img_p = None
            for ext in ('.JPG','.jpg','.jpeg','.png'):
                cand = img_dir / (txt.stem + ext)
                if cand.exists(): img_p = cand; break
            if img_p is None: continue
            img = cv2.imread(str(img_p))
            if img is None: continue
            H_orig, W = img.shape[:2]  # always use ORIGINAL dims
            boxes = []
            for line in txt.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5: continue
                try: ci,cx,cy,bw,bh = int(parts[0]),*[float(x) for x in parts[1:5]]
                except ValueError: continue
                cname = cls_names[ci] if ci < len(cls_names) else str(ci)
                y1 = (cy - bh/2) * H_orig
                y2 = (cy + bh/2) * H_orig
                # skip boxes entirely in OSD strip
                if strip_height > 0 and y1 >= (H_orig - strip_height): continue
                y2 = min(y2, H_orig - strip_height)
                boxes.append({'cls': cname,
                              'x1': (cx-bw/2)*W, 'y1': y1,
                              'x2': (cx+bw/2)*W, 'y2': y2})
            gt[str(img_p)] = boxes
            n_cam += len(boxes)
        if n_cam: print(f'  {cam_dir.name}: {n_cam} annotations')
    n_total = sum(len(v) for v in gt.values())
    cls_cnt = {}
    for boxes in gt.values():
        for b in boxes: cls_cnt[b['cls']] = cls_cnt.get(b['cls'],0)+1
    print(f'\n✓ GT: {len(gt)} images  {n_total} annotations')
    for c,n in sorted(cls_cnt.items()): print(f'  {c:15}: {n}')
    return gt

gt = load_gt(GT_ANN_ROOT, IMAGE_ROOT, GT_CLASSES, STRIP_HEIGHT)
gt_with_boxes = {k:v for k,v in gt.items() if len(v) > 0}
print(f'Images with annotations: {len(gt_with_boxes)}')


## Cell 5 — Load inference results

Loads all crop and YOLO results.
**Crop results**: every candidate bbox row (both insect AND background rejected).
**YOLO results**: every detection from yolo_results.csv.

In [ ]:
def detect_pipelines(csv_path):
    with open(csv_path, newline='') as f:
        fields = csv.DictReader(f).fieldnames or []
    return sorted({f.split('__')[0] for f in fields
                   if '__binary_label' in f or '__pollinator_type' in f})

def load_crop_run(run_path, image_root):
    run_path = Path(run_path)
    cfg_file = run_path / 'run_config.json'
    run_cfg  = _json.loads(cfg_file.read_text()) if cfg_file.exists() else {}
    rows = []; pipe_names = []
    for csv_path in sorted(run_path.rglob('results.csv')):
        cam_name = csv_path.parent.name
        img_dir  = Path(image_root) / cam_name
        if not pipe_names:
            pipe_names = detect_pipelines(csv_path)
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                img_name = row.get('image_name', '')
                # Build full path from camera_folder + image_name
                img_p = str(img_dir / img_name)
                if not Path(img_p).exists():
                    stem = Path(img_name).stem
                    for ext in ('.JPG','.jpg','.jpeg'):
                        cand = str(img_dir / (stem+ext))
                        if Path(cand).exists(): img_p=cand; break
                row['_img_path'] = img_p
                row['_cam']      = cam_name
                rows.append(row)
    pre = run_cfg.get('preprocess', {})
    print(f'  Pipelines : {pipe_names}')
    print(f'  Rows      : {len(rows)}')
    print(f'  Config    : large_motion={pre.get("enable_large_motion","?")}  '
          f'darker_threshold={pre.get("darker_threshold","?")}')
    return rows, pipe_names, run_cfg

def load_yolo_run(run_path, image_root):
    run_path = Path(run_path)
    cfg_file = run_path / 'run_config.json'
    run_cfg  = _json.loads(cfg_file.read_text()) if cfg_file.exists() else {}
    rows = []
    for csv_path in sorted(run_path.rglob('yolo_results.csv')):
        cam_name = csv_path.parent.name
        img_dir  = Path(image_root) / cam_name
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                img_name = row.get('image_name', '')
                img_p = str(img_dir / img_name)
                if not Path(img_p).exists():
                    stem = Path(img_name).stem
                    for ext in ('.JPG','.jpg','.jpeg'):
                        cand = str(img_dir / (stem+ext))
                        if Path(cand).exists(): img_p=cand; break
                row['_img_path'] = img_p
                row['_cam']      = cam_name
                rows.append(row)
    print(f'  YOLO detections: {len(rows)}')
    return rows, run_cfg

print('Loading crop runs...')
crop_run_data = {}
for name, path in CROP_RUNS.items():
    print(f'\n  {name}:')
    rows, pipes, cfg = load_crop_run(path, IMAGE_ROOT)
    crop_run_data[name] = {'rows':rows,'pipes':pipes,'config':cfg}

print('\nLoading YOLO runs...')
yolo_run_data = {}
for name, path in YOLO_RUNS.items():
    print(f'\n  {name}:')
    rows, cfg = load_yolo_run(path, IMAGE_ROOT)
    yolo_run_data[name] = {'rows':rows,'config':cfg}

print('\n✓ All runs loaded.')


## Cell 6 — Matching + metrics functions

**Do not edit.** Defines all evaluation logic.

**`center_match`**: matches predictions to GT using three criteria (any one sufficient):
1. GT center falls inside pred bbox
2. Pred center falls inside GT bbox
3. Overlap area / GT area ≥ 20%

**`evaluate_one_pipeline`**: for one pipeline computes:
- **TP** — bbox matched GT (any criterion above) + correct class
- **FP** — bbox with no matching GT (false alarm)
- **FN** — GT bbox with no matching prediction (missed insect)
- **FN breakdown**:
  - `fn_detected_as_bg` — pipeline HAD a bbox near this insect but classified it as background
  - `fn_not_detected` — no bbox at all near this insect (frame-diff completely missed it)
- **bg_rejected** — total candidate crops classified as background


In [ ]:
def bbox_overlap_ratio(p, g):
    ix1=max(p['x1'],g['x1']); iy1=max(p['y1'],g['y1'])
    ix2=min(p['x2'],g['x2']); iy2=min(p['y2'],g['y2'])
    iw=max(0,ix2-ix1); ih=max(0,iy2-iy1)
    gt_area=max(1,(g['x2']-g['x1'])*(g['y2']-g['y1']))
    return iw*ih/gt_area

def center_match(pred_boxes, gt_boxes, overlap_thresh=0.20):
    """Three criteria: GT center in pred, pred center in GT, or 20% overlap."""
    def inside(px,py,x1,y1,x2,y2): return x1<=px<=x2 and y1<=py<=y2
    mp=set(); mg=set(); pairs=[]
    for gi,g in enumerate(gt_boxes):
        gcx=(g['x1']+g['x2'])/2; gcy=(g['y1']+g['y2'])/2
        for pi,p in enumerate(pred_boxes):
            if pi in mp: continue
            pcx=(p['x1']+p['x2'])/2; pcy=(p['y1']+p['y2'])/2
            if (inside(gcx,gcy,p['x1'],p['y1'],p['x2'],p['y2']) or
                inside(pcx,pcy,g['x1'],g['y1'],g['x2'],g['y2']) or
                bbox_overlap_ratio(p,g)>=overlap_thresh):
                pairs.append((pi,gi)); mp.add(pi); mg.add(gi); break
    return pairs,[i for i in range(len(pred_boxes)) if i not in mp],\
                 [i for i in range(len(gt_boxes)) if i not in mg]

def evaluate_one_pipeline(preds_by_img, gt, classes, label):
    """
    Two-level evaluation:
    1. Detection: did pipeline find a bbox near the insect? (class-agnostic)
    2. Classification: was the class correct? (among detected)

    Also tracks:
    - fn_detected_as_bg: GT insects that WERE detected but classified as background
    - fn_not_detected:   GT insects that had NO bbox near them at all
    """
    rows_out=[]; n_bg_rejected=0
    det_tp=det_fp=det_fn=0
    cls_correct=0; cls_wrong=0
    tp_c=defaultdict(int); fp_c=defaultdict(int); fn_c=defaultdict(int)
    cls_confusion=defaultdict(lambda: defaultdict(int))

    # FN breakdown
    fn_detected_as_bg=0   # had a bbox but was rejected as background
    fn_not_detected=0     # no bbox at all near this GT insect

    for img_p, gt_boxes in gt.items():
        all_preds = preds_by_img.get(img_p, [])
        insect    = [p for p in all_preds if not p.get('is_bg')]
        rejected  = [p for p in all_preds if p.get('is_bg')]
        n_bg_rejected += len(rejected)

        # Match insect predictions to GT
        pairs, unp, ung = center_match(insect, gt_boxes)

        for pi,gi in pairs:
            det_tp+=1
            pc=insect[pi]['cls']; gc=gt_boxes[gi]['cls']
            correct=(pc==gc)
            if correct: cls_correct+=1; tp_c[gc]+=1
            else: cls_wrong+=1; fp_c[pc]+=1; fn_c[gc]+=1
            cls_confusion[gc][pc]+=1
            rows_out.append({'pipeline':label,'img':img_p,'match':'tp',
                             'pred_cls':pc,'gt_cls':gc,
                             'conf':insect[pi]['conf'],'correct_cls':correct})
        for pi in unp:
            det_fp+=1; fp_c[insect[pi]['cls']]+=1
            rows_out.append({'pipeline':label,'img':img_p,'match':'fp',
                             'pred_cls':insect[pi]['cls'],'gt_cls':'',
                             'conf':insect[pi]['conf'],'correct_cls':False})

        # For each unmatched GT, check if a REJECTED bbox covers it
        for gi in ung:
            det_fn+=1; fn_c[gt_boxes[gi]['cls']]+=1
            g = gt_boxes[gi]
            # Check if any bg_rejected bbox overlaps this GT
            covered_by_bg = any(
                bbox_overlap_ratio(r, g) >= 0.20 or
                bbox_overlap_ratio(g, r) >= 0.20
                for r in rejected
            )
            if covered_by_bg:
                fn_detected_as_bg+=1
                match_type='fn_detected_as_bg'
            else:
                fn_not_detected+=1
                match_type='fn_not_detected'
            rows_out.append({'pipeline':label,'img':img_p,'match':match_type,
                             'pred_cls':'bg','gt_cls':gt_boxes[gi]['cls'],
                             'conf':0.0,'correct_cls':False})

    det_prec=det_tp/max(1,det_tp+det_fp)
    det_rec =det_tp/max(1,det_tp+det_fn)
    det_f1  =2*det_prec*det_rec/max(1e-8,det_prec+det_rec)
    cls_acc =cls_correct/max(1,det_tp)

    print(f'  [Detection]      P={det_prec:.3f}  R={det_rec:.3f}  F1={det_f1:.3f}  '
          f'TP={det_tp}  FP={det_fp}  FN={det_fn}  bg_rejected={n_bg_rejected}')
    print(f'  [FN breakdown]   detected_as_bg={fn_detected_as_bg}  '
          f'not_detected={fn_not_detected}  '
          f'({100*fn_detected_as_bg/max(1,det_fn):.1f}% were detected but rejected)')
    print(f'  [Classification] accuracy={cls_acc:.3f}  '
          f'correct={cls_correct}  wrong={cls_wrong}  (of {det_tp} detected)')
    print(f'  Per-class:')
    for c in classes:
        if tp_c[c] or fp_c.get(c) or fn_c[c]:
            print(f'    {c:15} TP={tp_c[c]:>4}  FP={fp_c.get(c,0):>4}  FN={fn_c[c]:>4}')

    return {'det_precision':det_prec,'det_recall':det_rec,'det_f1':det_f1,
            'det_tp':det_tp,'det_fp':det_fp,'det_fn':det_fn,
            'fn_detected_as_bg':fn_detected_as_bg,
            'fn_not_detected':fn_not_detected,
            'cls_accuracy':cls_acc,'cls_correct':cls_correct,'cls_wrong':cls_wrong,
            'n_bg_rejected':n_bg_rejected,
            'tp_c':dict(tp_c),'fp_c':dict(fp_c),'fn_c':dict(fn_c),
            'cls_confusion':dict(cls_confusion),
            'rows':rows_out}


## Cell 7 — Build prediction index

Organises all predictions by image path for efficient matching.

For crop pipelines: every candidate bbox is included, both insect and background predictions.
`is_bg=True` means the pipeline classified this bbox as background.

For YOLO: every detection row from `yolo_results.csv`.


In [ ]:
def build_crop_index(rows, pipe_name):
    """Build {img_path -> list of pred dicts} for one pipeline."""
    idx = defaultdict(list)
    for r in rows:
        if r.get('pollinator_detected') not in ('yes',): continue
        try:
            x=int(r['bbox_x']); y=int(r['bbox_y'])
            w=int(r['bbox_w']); h=int(r['bbox_h'])
        except (ValueError,KeyError): continue
        p = pipe_name + '__'
        bl  = r.get(p+'binary_label', '')
        pt  = r.get(p+'pollinator_type', '')
        # Determine if this bbox is insect or background for this pipeline
        is_bg = (bl == 'background') or (pt == 'background') or \
                (not bl and not pt)
        pred_cls = pt if (pt and pt != 'background') else (bl if bl == 'insect' else 'background')
        try: conf = float(r.get(p+'group_conf') or r.get(p+'binary_conf') or 0)
        except: conf = 0.0
        idx[r['_img_path']].append({
            'x1':x,'y1':y,'x2':x+w,'y2':y+h,
            'cls':pred_cls,'conf':conf,'is_bg':is_bg
        })
    return idx

def build_yolo_index(rows):
    idx = defaultdict(list)
    for r in rows:
        try:
            x=int(r['bbox_x']); y=int(r['bbox_y'])
            w=int(r['bbox_w']); h=int(r['bbox_h'])
            conf=float(r.get('confidence',0))
        except (ValueError,KeyError): continue
        idx[r['_img_path']].append({
            'x1':x,'y1':y,'x2':x+w,'y2':y+h,
            'cls':r.get('class_name',''),'conf':conf,'is_bg':False
        })
    return idx

# Build all indexes
pred_indexes = {}  # label -> {img_path -> [preds]}

for run_name, run_data in crop_run_data.items():
    for pipe_name in run_data['pipes']:
        label = f'{run_name}/{pipe_name}'
        pred_indexes[label] = build_crop_index(run_data['rows'], pipe_name)
        n = sum(len(v) for v in pred_indexes[label].values())
        n_ins = sum(1 for v in pred_indexes[label].values() for p in v if not p['is_bg'])
        n_bg  = sum(1 for v in pred_indexes[label].values() for p in v if p['is_bg'])
        print(f'  {label}: {n} total  insect={n_ins}  background={n_bg}')

for run_name, run_data in yolo_run_data.items():
    label = run_name
    pred_indexes[label] = build_yolo_index(run_data['rows'])
    n = sum(len(v) for v in pred_indexes[label].values())
    print(f'  {label}: {n} detections')

print(f'\n✓ Prediction indexes built for {len(pred_indexes)} pipelines.')


## Cell 8 — Run evaluation  ← main evaluation cell

Evaluates every pipeline against GT. Prints results as it goes.


In [ ]:
all_results = {}

for label, preds_by_img in pred_indexes.items():
    print(f'\n=== {label} ===')
    all_results[label] = evaluate_one_pipeline(
        preds_by_img, gt, GT_CLASSES, label)

print(f'\n✓ Evaluated {len(all_results)} pipelines.')

# ── Additional: all pipelines evaluated excluding bumblebee ──────
# (fair comparison since YOLO was not trained on bumblebee)
print('\n' + '='*70)
print('All pipelines — bumblebee EXCLUDED from GT (fair YOLO comparison)')
print('='*70)

CLASSES_NO_BB = ['fly', 'butterfly', 'other']
gt_no_bb = {img_p: [b for b in boxes if b['cls'] in CLASSES_NO_BB]
            for img_p, boxes in gt.items()}

for label, preds_by_img in pred_indexes.items():
    print(f'\n=== {label} ===')
    all_results[label + '_no_bb'] = evaluate_one_pipeline(
        preds_by_img, gt_no_bb, CLASSES_NO_BB, label + '_no_bb')


## Cell 9 — Results table + confusion matrix + save

Prints:
1. **Overall detection metrics** (class-agnostic P/R/F1) with FN breakdown
2. **Per-class P/R/F1**
3. **Classification accuracy** among detected insects
4. **Confusion matrix** (GT class vs predicted class, including bg_rejected and missed)

Saves `eval_results.csv` and `summary.json` to `evaluation/`.


In [ ]:
_start_capture('overall_metrics')
import csv as _csv
import json as _json
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

# Split results into two groups for cleaner display
_regular = {k: v for k, v in all_results.items() if not k.endswith('_no_bb')}
_no_bb   = {k: v for k, v in all_results.items() if k.endswith('_no_bb')}

def _print_overall_table(results):
    print(f'{"Pipeline":40}  {"Precision":>10}  {"Recall":>7}  {"F1_Score":>9}  {"True_Pos":>9}  {"False_Pos":>10}  {"False_Neg":>10}  {"BG_Rejected":>12}')
    print('-'*105)
    for label, r in results.items():
        print(f'{label:40}  {r["det_precision"]:>10.3f}  {r["det_recall"]:>7.3f}  '
              f'{r["det_f1"]:>9.3f}  {r["det_tp"]:>9}  {r["det_fp"]:>10}  '
              f'{r["det_fn"]:>10}  {r["n_bg_rejected"]:>12}')

def _print_perclass_table(results, classes):
    for label, r in results.items():
        print(f'\n  {label}')
        print(f'  {"Class":15}  {"Precision":>10}  {"Recall":>7}  {"F1_Score":>9}  {"True_Pos":>9}  {"False_Pos":>10}  {"False_Neg":>10}')
        print(f'  {"-"*75}')
        for c in classes:
            tp = r["tp_c"].get(c, 0)
            fp = r["fp_c"].get(c, 0)
            fn = r["fn_c"].get(c, 0)
            p  = tp / max(1, tp+fp)
            rc = tp / max(1, tp+fn)
            f1 = 2*p*rc / max(1e-8, p+rc)
            print(f'  {c:15}  {p:>10.3f}  {rc:>7.3f}  {f1:>9.3f}  {tp:>9}  {fp:>10}  {fn:>10}')

def _print_cls_accuracy(results):
    print(f'{"Pipeline":40}  {"Accuracy":>9}  {"Correct":>8}  {"Wrong":>7}')
    print('-'*70)
    for label, r in results.items():
        print(f'{label:40}  {r["cls_accuracy"]:>9.3f}  {r["cls_correct"]:>8}  {r["cls_wrong"]:>7}')

def _print_confusion(results, classes):
    col_w = 20
    cols  = classes + ['background_rejected', 'missed']
    _hdr  = "Ground Truth / Prediction"
    header = f'  {_hdr:26}' + ''.join(f'{c:>{col_w}}' for c in cols)
    for label, r in results.items():
        print(f'\n  {label}')
        matrix = defaultdict(lambda: defaultdict(int))
        missed = defaultdict(int)
        bg_rej = defaultdict(int)
        for row in r['rows']:
            if row['match'] == 'tp':
                matrix[row['gt_cls']][row['pred_cls']] += 1
            elif row['match'] == 'fn':
                missed[row['gt_cls']] += 1
        for c in classes:
            bg_rej[c] = r['fn_c'].get(c, 0) - missed[c]
            if bg_rej[c] < 0: bg_rej[c] = 0
        print(header)
        print('  ' + '-'*(26 + col_w*len(cols)))
        for gt_c in classes:
            row_str = f'  {gt_c:26}'
            for pred_c in classes:
                row_str += f'{matrix[gt_c][pred_c]:>{col_w}}'
            row_str += f'{bg_rej[gt_c]:>{col_w}}'
            row_str += f'{missed[gt_c]:>{col_w}}'
            print(row_str)

# ── Section 1: all 4 classes (includes bumblebee) ────────────────
print('\n' + '█'*105)
print('  ALL CLASSES  (bumblebee · fly · butterfly · other)')
print('█'*105)

print('\n' + '='*105)
print('Overall Detection Metrics')
print('='*105)
_print_overall_table(_regular)

print('\n' + '='*105)
print('Per-class Detection Metrics')
print('='*105)
_print_perclass_table(_regular, GT_CLASSES)

print('\n' + '='*80)
print('Classification Accuracy (among detected insects)')
print('='*80)
_print_cls_accuracy(_regular)

print('\n' + '='*80)
print('Confusion Matrices')
print('='*80)
_print_confusion(_regular, GT_CLASSES)

# ── Section 2: bumblebee excluded (fair YOLO comparison) ─────────
print('\n\n' + '█'*105)
print('  BUMBLEBEE EXCLUDED  (fly · butterfly · other) — fair comparison since YOLO was not trained on bumblebee')
print('█'*105)

print('\n' + '='*105)
print('Overall Detection Metrics')
print('='*105)
_print_overall_table(_no_bb)

print('\n' + '='*105)
print('Per-class Detection Metrics')
print('='*105)
_print_perclass_table(_no_bb, CLASSES_NO_BB)

print('\n' + '='*80)
print('Classification Accuracy (among detected insects)')
print('='*80)
_print_cls_accuracy(_no_bb)

print('\n' + '='*80)
print('Confusion Matrices')
print('='*80)
_print_confusion(_no_bb, CLASSES_NO_BB)

# ── Build xlsx data ───────────────────────────────────────────────
_xls_overall = []
for label, r in all_results.items():
    _xls_overall.append({
        'pipeline':               label,
        'bumblebee_excluded':     label.endswith('_no_bb'),
        'precision':              round(r['det_precision'], 4),
        'recall':                 round(r['det_recall'],    4),
        'f1_score':               round(r['det_f1'],        4),
        'true_positives':         r['det_tp'],
        'false_positives':        r['det_fp'],
        'false_negatives':        r['det_fn'],
        'fn_detected_as_background': r['fn_detected_as_bg'],
        'fn_not_detected':        r['fn_not_detected'],
        'classification_accuracy': round(r['cls_accuracy'], 4),
        'correct':                r['cls_correct'],
        'wrong':                  r['cls_wrong'],
        'background_rejected':    r['n_bg_rejected'],
    })

_xls_per_class = []
for label, r in all_results.items():
    classes = CLASSES_NO_BB if label.endswith('_no_bb') else GT_CLASSES
    for c in classes:
        tp = r['tp_c'].get(c, 0); fp = r['fp_c'].get(c, 0); fn = r['fn_c'].get(c, 0)
        p = tp/max(1,tp+fp); rc = tp/max(1,tp+fn); f1 = 2*p*rc/max(1e-8,p+rc)
        _xls_per_class.append({
            'pipeline':            label,
            'bumblebee_excluded':  label.endswith('_no_bb'),
            'class':               c,
            'precision':           round(p,  4),
            'recall':              round(rc, 4),
            'f1_score':            round(f1, 4),
            'true_positives':      tp,
            'false_positives':     fp,
            'false_negatives':     fn,
        })

_xls_confusion = []
_cm_cnt = Counter()
for r in all_results.values():
    for row in r['rows']:
        if row['match'] == 'tp':
            _cm_cnt[(row['pipeline'], row['gt_cls'], row['pred_cls'])] += 1
        elif row['match'] in ('fn_detected_as_bg', 'fn_not_detected'):
            _col = 'background_rejected' if row['match'] == 'fn_detected_as_bg' else 'missed'
            _cm_cnt[(row['pipeline'], row['gt_cls'], _col)] += 1
for (pipe, gt_c, pred_c), cnt in sorted(_cm_cnt.items()):
    _xls_confusion.append({
        'pipeline':           pipe,
        'ground_truth_class': gt_c,
        'predicted_class':    pred_c,
        'count':              cnt,
    })

_xls_raw = []
for r in all_results.values():
    for row in r['rows']:
        _xls_raw.append({
            'pipeline':           row['pipeline'],
            'image':              row['img'],
            'match_type':         row['match'],
            'predicted_class':    row['pred_cls'],
            'ground_truth_class': row['gt_cls'],
            'confidence':         row['conf'],
            'correct_class':      row['correct_cls'],
        })

print(f'\n\u2713 Metrics collected (will be saved to xlsx at end of Cell 14)')
_stop_capture('overall_metrics')


## Cell 10 — Confidence Threshold Analysis

For each pipeline, shows how P/R/F1 change at different confidence thresholds.

**Goal:** find the threshold that achieves target recall (≥0.8) with best precision.
Since the goal is not to miss insects, we prioritise recall.

Saves `threshold_analysis.png` to `evaluation/`.


In [ ]:
# ── Threshold analysis ────────────────────────────────────────────
RECALL_TARGETS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

print('\n' + '='*75)
print('Threshold Analysis — Recall-constrained')
print('(find best threshold for each recall target)')
print('='*75)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = plt.cm.tab10.colors

for ci, (label, r) in enumerate(all_results.items()):
    color = colors[ci % len(colors)]

    # Get all (conf, is_tp) pairs from rows
    dets = sorted(
        [(row['conf'], row['match'] == 'tp')
         for row in r['rows'] if row['match'] in ('tp', 'fp')],
        key=lambda x: -x[0])
    if not dets: continue

    n_gt = r['det_tp'] + r['det_fn']
    thresholds=[]; precisions=[]; recalls=[]; f1s=[]
    tp_ = fp_ = 0

    for conf, is_tp in dets:
        if is_tp: tp_ += 1
        else: fp_ += 1
        p  = tp_ / max(1, tp_+fp_)
        rc = tp_ / max(1, n_gt)
        f1 = 2*p*rc / max(1e-8, p+rc)
        thresholds.append(conf)
        precisions.append(p)
        recalls.append(rc)
        f1s.append(f1)

    # PR curve
    axes[0].plot(recalls, precisions, color=color,
                 label=f'{label} (F1={r["det_f1"]:.3f})', lw=2)

    # Threshold vs Recall/Precision
    axes[1].plot(thresholds, recalls, color=color, ls='-', lw=2,
                 label=f'{label} recall')
    axes[1].plot(thresholds, precisions, color=color, ls='--', lw=1,
                 alpha=0.6)

    # Find best threshold for each recall target
    print(f'\n  {label}')
    print(f'  {"Recall target":15}  {"Threshold":>10}  {"Precision":>10}  {"F1":>8}')
    print(f'  {"-"*50}')
    for target in RECALL_TARGETS:
        # Find highest threshold that achieves this recall
        best_thr = best_p = best_f1 = None
        for thr, p, rc, f1 in zip(thresholds, precisions, recalls, f1s):
            if rc >= target:
                if best_thr is None or thr > best_thr:
                    best_thr = thr; best_p = p; best_f1 = f1
        if best_thr is not None:
            print(f'  R≥{target:.2f}          {best_thr:>10.3f}  {best_p:>10.3f}  {best_f1:>8.3f}')
        else:
            print(f'  R≥{target:.2f}          {"N/A":>10}  {"N/A":>10}  {"N/A":>8}')

    # Mark optimal F1 point
    best_idx = int(np.argmax(f1s))
    axes[0].plot(recalls[best_idx], precisions[best_idx],
                 'o', color=color, ms=8)
    axes[0].annotate(f'  thr={thresholds[best_idx]:.2f}',
                     (recalls[best_idx], precisions[best_idx]),
                     fontsize=7, color=color)

axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('PR Curve (● = best F1)')
axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0,1); axes[0].set_ylim(0,1)

axes[1].set_xlabel('Confidence Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Recall (—) and Precision (--) vs Threshold')
axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)

# Mark recall=0.8 line
axes[0].axhline(y=0, color='grey', ls=':', alpha=0.5)
axes[1].axhline(y=0.8, color='grey', ls=':', lw=1, label='R=0.8 target')

plt.suptitle('Confidence Threshold Analysis', fontsize=13)
plt.tight_layout()
plt.savefig(EVAL_DIR/'threshold_analysis.png', dpi=150)
print(f'\n✓ Saved threshold_analysis.png')
plt.show()

# ── Collect threshold data for xlsx ───────────────────────────────
_xls_threshold = []
for _label, _r in all_results.items():
    _dets = sorted(
        [(_row['conf'], _row['match'] == 'tp')
         for _row in _r['rows'] if _row['match'] in ('tp', 'fp')],
        key=lambda x: -x[0])
    if not _dets: continue
    _n_gt = _r['det_tp'] + _r['det_fn']
    _tp2 = _fp2 = 0
    for _conf, _is_tp in _dets:
        if _is_tp: _tp2 += 1
        else: _fp2 += 1
        _p2  = _tp2 / max(1, _tp2 + _fp2)
        _rc2 = _tp2 / max(1, _n_gt)
        _f12 = 2 * _p2 * _rc2 / max(1e-8, _p2 + _rc2)
        _xls_threshold.append({'pipeline':_label,
            'threshold':round(_conf,4),'precision':round(_p2,4),
            'recall':round(_rc2,4),'f1':round(_f12,4),'tp':_tp2,'fp':_fp2})


## Cell 11 — Browse images with all pipeline bboxes

Displays original images with GT + all pipeline bboxes drawn in memory.
**No files are saved.** Uses matplotlib for cross-platform keyboard navigation.

**Keys:** ← → to navigate, q to quit.

Run `%matplotlib tk` in a separate cell first if using locally.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2, numpy as np
from pathlib import Path
from collections import defaultdict

# ── Build display index: img_path -> list of (label,x1,y1,x2,y2,color,thick) ──
display_index = defaultdict(list)

COLORS = {
    'GT':             (0,   0.78, 0),      # green
    'two_stage':      (1,   0.39, 0),      # blue-ish orange
    'five_class_eff': (0,   0.55, 1),      # orange
    'five_class_ins': (0.7, 0,   1),       # purple
    'yolo_run_01':    (1,   0,   0),       # red
}
DEFAULT_COLOR = (0.5, 0.5, 0.5)

# GT
for img_p, boxes in gt_with_boxes.items():
    for b in boxes:
        display_index[img_p].append(
            (f'GT:{b["cls"]}', b['x1'],b['y1'],b['x2'],b['y2'],
             COLORS['GT'], 2))

# All pipelines
for label, preds_by_img in pred_indexes.items():
    pipe_key = label.split('/')[-1] if '/' in label else label
    color = COLORS.get(pipe_key, DEFAULT_COLOR)
    for img_p, preds in preds_by_img.items():
        for p in preds:
            is_bg = p.get('is_bg', False)
            thick = 0.5 if is_bg else 2
            lbl   = '' if is_bg else f'{pipe_key}:{p["cls"]}'
            c     = (0.7,0.7,0.7) if is_bg else color
            display_index[img_p].append(
                (lbl, p['x1'],p['y1'],p['x2'],p['y2'], c, thick))

img_list = [p for p in sorted(display_index.keys()) if display_index[p]]
print(f'{len(img_list)} images with detections or GT')
print('Keys: ← → to navigate, q to quit')

idx = [0]
fig, ax = plt.subplots(figsize=(16, 9))
plt.subplots_adjust(top=0.95, bottom=0.02)

def draw(i):
    ax.clear()
    img_p = img_list[i]
    img   = cv2.imread(img_p)
    if img is None: return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    for lbl,x1,y1,x2,y2,color,thick in display_index[img_p]:
        rect = mpatches.Rectangle(
            (x1,y1), x2-x1, y2-y1,
            linewidth=thick, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        if lbl:
            ax.text(x1, max(y1-4,10), lbl,
                    color=color, fontsize=6,
                    bbox=dict(fc='black', alpha=0.4, pad=1, ec='none'))
    name = f'{Path(img_p).parent.name}/{Path(img_p).name}'
    ax.set_title(f'[{i+1}/{len(img_list)}]  {name}', fontsize=9)
    ax.axis('off')
    fig.canvas.draw_idle()

def on_key(event):
    if event.key == 'right':
        idx[0] = min(len(img_list)-1, idx[0]+1); draw(idx[0])
    elif event.key == 'left':
        idx[0] = max(0, idx[0]-1); draw(idx[0])
    elif event.key == 'q':
        plt.close()

fig.canvas.mpl_connect('key_press_event', on_key)
draw(0)
plt.show()


## Cell 12 — Per-plot (per camera folder) metrics

Shows detection metrics broken down by each camera folder.
Useful for spotting which plots are harder for each pipeline.


In [ ]:
from collections import defaultdict as _dd12

print('\n' + '='*100)
print('Per-plot Detection Metrics')
print('='*100)

_xls_plot_det = []
_xls_plot_cls = []

_start_capture('per_plot_metrics')

def _perplot_section(pred_indexes_subset, section_classes):
    for label, preds_by_img in pred_indexes_subset.items():
        print(f'\n{"="*70}')
        print(f'  {label}')
        print(f'{"="*70}')

        plots = {}
        for img_p, gt_boxes in gt.items():
            cam = Path(img_p).parent.name
            plots.setdefault(cam, {'gt': {}, 'preds': {}})
            plots[cam]['gt'][img_p] = gt_boxes
        for img_p, preds in preds_by_img.items():
            cam = Path(img_p).parent.name
            if cam in plots:
                plots[cam]['preds'][img_p] = preds

        print(f'\n  [Detection — class agnostic]')
        print(f'  {"Plot":45}  {"Precision":>10}  {"Recall":>7}  {"F1_Score":>9}  {"True_Pos":>9}  {"False_Pos":>10}  {"False_Neg":>10}  {"Ground_Truth":>13}')
        print(f'  {"-"*115}')
        for cam in sorted(plots.keys()):
            cam_gt    = plots[cam]['gt']
            cam_preds = plots[cam]['preds']
            tp=fp=fn=0
            for img_p, gt_boxes in cam_gt.items():
                insect = [p for p in cam_preds.get(img_p,[]) if not p.get('is_bg')]
                pairs, unp, ung = center_match(insect, gt_boxes)
                tp+=len(pairs); fp+=len(unp); fn+=len(ung)
            p=tp/max(1,tp+fp); r=tp/max(1,tp+fn); f1=2*p*r/max(1e-8,p+r)
            print(f'  {cam:45}  {p:>10.3f}  {r:>7.3f}  {f1:>9.3f}  {tp:>9}  {fp:>10}  {fn:>10}  {tp+fn:>13}')
            _xls_plot_det.append({
                'pipeline':             label,
                'bumblebee_excluded':   label.endswith('_no_bb'),
                'plot':                 cam,
                'precision':            round(p, 4),
                'recall':               round(r, 4),
                'f1_score':             round(f1, 4),
                'true_positives':       tp,
                'false_positives':      fp,
                'false_negatives':      fn,
                'ground_truth_total':   tp+fn,
            })

        print(f'\n  [Per-class breakdown per plot]')
        print(f'  {"Plot":45}  {"Class":12}  {"Precision":>10}  {"Recall":>7}  {"F1_Score":>9}  {"True_Pos":>9}  {"False_Pos":>10}  {"False_Neg":>10}  {"Accuracy":>9}  {"Correct":>8}  {"Wrong":>6}')
        print(f'  {"-"*130}')

        for cam in sorted(plots.keys()):
            cam_gt    = plots[cam]['gt']
            cam_preds = plots[cam]['preds']
            # Use defaultdict so unknown predicted classes don't cause KeyError
            cls_tp      = _dd12(int)
            cls_fp      = _dd12(int)
            cls_fn      = _dd12(int)
            cls_correct = _dd12(int)
            cls_wrong   = _dd12(int)
            cls_errors  = _dd12(lambda: _dd12(int))

            for img_p, gt_boxes in cam_gt.items():
                insect = [p for p in cam_preds.get(img_p,[]) if not p.get('is_bg')]
                pairs, unp, ung = center_match(insect, gt_boxes)
                for pi,gi in pairs:
                    pc=insect[pi]['cls']; gc=gt_boxes[gi]['cls']
                    if pc==gc: cls_tp[gc]+=1; cls_correct[gc]+=1
                    else: cls_tp[gc]+=1; cls_wrong[gc]+=1; cls_errors[gc][pc]+=1; cls_fp[pc]+=1
                for pi in unp: cls_fp[insect[pi]['cls']]+=1
                for gi in ung: cls_fn[gt_boxes[gi]['cls']]+=1

            first = True
            for c in section_classes:
                tp=cls_tp[c]; fp=cls_fp[c]; fn=cls_fn[c]
                correct=cls_correct[c]; wrong=cls_wrong[c]
                if tp==0 and fp==0 and fn==0: continue
                p=tp/max(1,tp+fp); r=tp/max(1,tp+fn); f1=2*p*r/max(1e-8,p+r)
                acc=correct/max(1,tp)
                cam_label = cam if first else ''
                print(f'  {cam_label:45}  {c:12}  {p:>10.3f}  {r:>7.3f}  {f1:>9.3f}  {tp:>9}  {fp:>10}  {fn:>10}  {acc:>9.3f}  {correct:>8}  {wrong:>6}')
                if cls_errors[c]:
                    err_str = '  '.join(f'{gc}\u2192{pc}:{n}' for pc,n in sorted(cls_errors[c].items(), key=lambda x:-x[1]))
                    print(f'  {"":45}  {"  misclassified:":14}  {err_str}')
                first = False
                _xls_plot_cls.append({
                    'pipeline':               label,
                    'bumblebee_excluded':     label.endswith('_no_bb'),
                    'plot':                   cam,
                    'class':                  c,
                    'precision':              round(p,   4),
                    'recall':                 round(r,   4),
                    'f1_score':               round(f1,  4),
                    'true_positives':         tp,
                    'false_positives':        fp,
                    'false_negatives':        fn,
                    'classification_accuracy': round(acc, 4),
                    'correct':                correct,
                    'wrong':                  wrong,
                })
            if not first: print()

_regular_idx = {k: v for k, v in pred_indexes.items() if not k.endswith('_no_bb')}
_no_bb_idx   = {k: v for k, v in pred_indexes.items() if k.endswith('_no_bb')}

print('\n' + '█'*100)
print('  ALL CLASSES  (bumblebee · fly · butterfly · other)')
print('█'*100)
_perplot_section(_regular_idx, GT_CLASSES)

print('\n\n' + '█'*100)
print('  BUMBLEBEE EXCLUDED  (fly · butterfly · other)')
print('█'*100)
_perplot_section(_no_bb_idx, CLASSES_NO_BB)

_stop_capture('per_plot_metrics')


## Cell 13 — Combined pipeline analysis (crop + YOLO)

For each crop pipeline paired with YOLO, shows:
- How many GT insects each pipeline detects alone
- How many are detected by BOTH (overlap)
- Combined recall (union of detections)
- Among insects detected by both: classification agreement


In [ ]:
_start_capture('combined_analysis')
print('\n' + '='*90)
print('Combined Pipeline Analysis (crop + YOLO)')
print('='*90)

YOLO_LABEL = 'yolo_run_01'
yolo_preds = pred_indexes.get(YOLO_LABEL, {})

crop_labels = [l for l in pred_indexes if l != YOLO_LABEL]

for crop_label in crop_labels:
    crop_preds = pred_indexes[crop_label]
    print(f'\n{"="*70}')
    print(f'  {crop_label}  +  {YOLO_LABEL}')
    print(f'{"="*70}')

    # For each GT insect, check who detected it
    crop_only=0; yolo_only=0; both=0; neither=0
    agree=0; disagree=0
    combined_tp=0; combined_fp_crop=0; combined_fp_yolo=0

    # Track per-class combined recall
    cls_gt   = {c:0 for c in GT_CLASSES}
    cls_crop = {c:0 for c in GT_CLASSES}
    cls_yolo = {c:0 for c in GT_CLASSES}
    cls_both = {c:0 for c in GT_CLASSES}
    cls_union= {c:0 for c in GT_CLASSES}

    for img_p, gt_boxes in gt.items():
        if not gt_boxes: continue
        crop_ins = [p for p in crop_preds.get(img_p,[]) if not p.get('is_bg')]
        yolo_ins = yolo_preds.get(img_p, [])

        crop_pairs, _, crop_ung = center_match(crop_ins, gt_boxes)
        yolo_pairs, _, yolo_ung = center_match(yolo_ins, gt_boxes)

        crop_matched_gt = {gi for _,gi in crop_pairs}
        yolo_matched_gt = {gi for _,gi in yolo_pairs}

        for gi, g in enumerate(gt_boxes):
            gc = g['cls']
            cls_gt[gc] = cls_gt.get(gc,0) + 1
            in_crop = gi in crop_matched_gt
            in_yolo = gi in yolo_matched_gt

            if in_crop: cls_crop[gc] = cls_crop.get(gc,0) + 1
            if in_yolo: cls_yolo[gc] = cls_yolo.get(gc,0) + 1
            if in_crop or in_yolo:
                cls_union[gc] = cls_union.get(gc,0) + 1

            if in_crop and in_yolo:
                both += 1; cls_both[gc] = cls_both.get(gc,0) + 1
                # Check classification agreement
                crop_pi = next(pi for pi,gi2 in crop_pairs if gi2==gi)
                yolo_pi = next(pi for pi,gi2 in yolo_pairs if gi2==gi)
                cp = crop_ins[crop_pi]['cls']
                yp = yolo_ins[yolo_pi]['cls']
                if cp == yp == gc: agree += 1
                elif cp == gc or yp == gc: agree += 1  # at least one correct
                else: disagree += 1
            elif in_crop:
                crop_only += 1
            elif in_yolo:
                yolo_only += 1
            else:
                neither += 1

    total_gt = crop_only + yolo_only + both + neither
    combined_detected = crop_only + yolo_only + both
    combined_recall = combined_detected / max(1, total_gt)
    crop_recall  = (crop_only + both) / max(1, total_gt)
    yolo_recall  = (yolo_only + both) / max(1, total_gt)

    print(f'\n  GT insects total      : {total_gt}')
    print(f'  Detected by crop only : {crop_only:>4}  ({100*crop_only/max(1,total_gt):.1f}%)')
    print(f'  Detected by YOLO only : {yolo_only:>4}  ({100*yolo_only/max(1,total_gt):.1f}%)')
    print(f'  Detected by BOTH      : {both:>4}  ({100*both/max(1,total_gt):.1f}%)')
    print(f'  Detected by neither   : {neither:>4}  ({100*neither/max(1,total_gt):.1f}%)')
    print(f'')
    print(f'  Crop recall alone     : {crop_recall:.3f}')
    print(f'  YOLO recall alone     : {yolo_recall:.3f}')
    print(f'  Combined recall (union): {combined_recall:.3f}  (+{100*(combined_recall-max(crop_recall,yolo_recall))/max(0.001,max(crop_recall,yolo_recall)):.1f}% over best single)')
    print(f'')
    print(f'  Among detected by BOTH ({both}):')
    print(f'    Classification agree : {agree}')
    print(f'    Classification disagree: {disagree}')

    print(f'\n  Per-class combined recall:')
    print(f'  {"Class":15}  {"GT":>5}  {"crop":>6}  {"yolo":>6}  {"both":>6}  {"union":>6}  {"R_crop":>8}  {"R_yolo":>8}  {"R_union":>8}')
    print(f'  {"-"*75}')
    for c in GT_CLASSES:
        n  = cls_gt.get(c,0)
        nc = cls_crop.get(c,0)
        ny = cls_yolo.get(c,0)
        nb = cls_both.get(c,0)
        nu = cls_union.get(c,0)
        rc = nc/max(1,n); ry = ny/max(1,n); ru = nu/max(1,n)
        print(f'  {c:15}  {n:>5}  {nc:>6}  {ny:>6}  {nb:>6}  {nu:>6}  {rc:>8.3f}  {ry:>8.3f}  {ru:>8.3f}')

# ── Collect combined analysis for xlsx ─────────────────────────────
_xls_combined = []
for _crop_label in [_l for _l in pred_indexes if _l != YOLO_LABEL]:
    _cp = pred_indexes[_crop_label]; _yp = pred_indexes.get(YOLO_LABEL, {})
    _co=_yo=_bo=_ne=0
    _cg2={c:0 for c in GT_CLASSES}; _cc2={c:0 for c in GT_CLASSES}
    _cy2={c:0 for c in GT_CLASSES}; _cu2={c:0 for c in GT_CLASSES}
    for _img_p, _gt_boxes in gt.items():
        if not _gt_boxes: continue
        _ci=[p for p in _cp.get(_img_p,[]) if not p.get('is_bg')]
        _yi=_yp.get(_img_p,[])
        _cp3,_,_=center_match(_ci,_gt_boxes); _yp3,_,_=center_match(_yi,_gt_boxes)
        _cg3={gi for _,gi in _cp3}; _yg3={gi for _,gi in _yp3}
        for _gi,_g in enumerate(_gt_boxes):
            _gc=_g['cls']; _cg2[_gc]=_cg2.get(_gc,0)+1
            _ic=_gi in _cg3; _iy=_gi in _yg3
            if _ic: _cc2[_gc]=_cc2.get(_gc,0)+1
            if _iy: _cy2[_gc]=_cy2.get(_gc,0)+1
            if _ic or _iy: _cu2[_gc]=_cu2.get(_gc,0)+1
            if _ic and _iy: _bo+=1
            elif _ic: _co+=1
            elif _iy: _yo+=1
            else: _ne+=1
    _tot=_co+_yo+_bo+_ne
    # summary row
    _xls_combined.append({'crop_pipeline':_crop_label,'yolo_pipeline':YOLO_LABEL,
        'class':'__ALL__','total_gt':_tot,
        'only_crop':_co,'only_yolo':_yo,'detected_by_both':_bo,'detected_by_neither':_ne,
        'crop_recall':round((_co+_bo)/max(1,_tot),4),
        'yolo_recall':round((_yo+_bo)/max(1,_tot),4),
        'combined_recall':round((_co+_yo+_bo)/max(1,_tot),4)})
    for _c in GT_CLASSES:
        _n=_cg2.get(_c,0)
        _xls_combined.append({'crop_pipeline':_crop_label,'yolo_pipeline':YOLO_LABEL,
            'class':_c,'total_gt':_n,
            'crop_only':0,'yolo_only':0,'both':0,'neither':0,
            'recall_crop':round(_cc2.get(_c,0)/max(1,_n),4),
            'recall_yolo':round(_cy2.get(_c,0)/max(1,_n),4),
            'recall_union':round(_cu2.get(_c,0)/max(1,_n),4)})

_stop_capture('combined_analysis')


## Cell 14 — Unique detections per pipeline

For each pipeline: GT insects that ONLY this pipeline detected, no other pipeline found them.
Useful for understanding what each pipeline uniquely contributes.


In [ ]:
print('\n' + '='*70)
print('Unique Detections — insects only this pipeline found')
print('='*70)

all_labels = list(pred_indexes.keys())

# For each GT insect, record which pipelines detected it
gt_detected_by = {}  # (img_p, gi) -> set of pipeline labels

for label, preds_by_img in pred_indexes.items():
    for img_p, gt_boxes in gt.items():
        if not gt_boxes: continue
        insect = [p for p in preds_by_img.get(img_p,[]) if not p.get('is_bg')]
        pairs, _, _ = center_match(insect, gt_boxes)
        for _, gi in pairs:
            key = (img_p, gi)
            gt_detected_by.setdefault(key, set()).add(label)

# For each pipeline, count unique detections
print(f'\n  {"Pipeline":40}  {"Unique":>7}  {"% of GT":>8}  {"% of its TP":>12}')
print(f'  {"-"*70}')

total_gt = sum(1 for boxes in gt.values() for _ in boxes)

for label in all_labels:
    # TP for this pipeline
    tp_keys = {k for k,v in gt_detected_by.items() if label in v}
    # Unique: only this pipeline detected it
    unique_keys = {k for k,v in gt_detected_by.items()
                   if v == {label}}
    n_unique = len(unique_keys)
    n_tp     = len(tp_keys)
    pct_gt   = 100*n_unique/max(1,total_gt)
    pct_tp   = 100*n_unique/max(1,n_tp)
    print(f'  {label:40}  {n_unique:>7}  {pct_gt:>7.1f}%  {pct_tp:>11.1f}%')

# Per-class breakdown
print(f'\n  Per-class unique detections:')
print(f'  {"Pipeline":40}  ' + '  '.join(f'{c:>10}' for c in GT_CLASSES))
print(f'  {"-"*80}')

for label in all_labels:
    unique_keys = {k for k,v in gt_detected_by.items() if v == {label}}
    cls_counts = {c:0 for c in GT_CLASSES}
    for (img_p, gi) in unique_keys:
        gc = gt[img_p][gi]['cls']
        cls_counts[gc] = cls_counts.get(gc,0) + 1
    row = f'  {label:40}  ' + '  '.join(f'{cls_counts.get(c,0):>10}' for c in GT_CLASSES)
    print(row)

# Also show: insects detected by ALL pipelines
all_detected = {k for k,v in gt_detected_by.items() if len(v) == len(all_labels)}
print(f'\n  Detected by ALL {len(all_labels)} pipelines: {len(all_detected)} GT insects')
print(f'  Not detected by ANY pipeline: {total_gt - len(gt_detected_by)} GT insects')

_start_capture('unique_detections')
# ── Collect unique detections for xlsx ───────────────────────────
_xls_unique = []
for _label in all_labels:
    _tp_k = {k for k,v in gt_detected_by.items() if _label in v}
    _uq_k = {k for k,v in gt_detected_by.items() if v == {_label}}
    _cls_uq = {c:0 for c in GT_CLASSES}
    for (_ip,_gi) in _uq_k:
        _gc=gt[_ip][_gi]['cls']; _cls_uq[_gc]=_cls_uq.get(_gc,0)+1
    _row={'pipeline':_label,'uniquely_detected':len(_uq_k),
          'pipeline_true_positives':len(_tp_k),'total_ground_truth':total_gt,
          'pct_of_ground_truth':round(100*len(_uq_k)/max(1,total_gt),2),
          'pct_of_true_positives':round(100*len(_uq_k)/max(1,len(_tp_k)),2)}
    for _c in GT_CLASSES: _row[f'unique_{_c}']=_cls_uq.get(_c,0)
    _xls_unique.append(_row)

_stop_capture('unique_detections')
# ── Write everything to one xlsx ──────────────────────────────────────────
try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','install','openpyxl','-q'])
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

def _write_sheet(ws, rows, col_names=None):
    """Write a list-of-dicts to a worksheet with header row."""
    if not rows: return
    cols = col_names or list(rows[0].keys())
    HDR_FILL = PatternFill('solid', start_color='1F4E79')
    HDR_FONT = Font(bold=True, color='FFFFFF', name='Arial', size=10)
    ROW_FONT = Font(name='Arial', size=10)
    ALT_FILL = PatternFill('solid', start_color='D6E4F7')

    for ci, col in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=col)
        cell.font = HDR_FONT; cell.fill = HDR_FILL
        cell.alignment = Alignment(horizontal='center')

    for ri, row in enumerate(rows, 2):
        for ci, col in enumerate(cols, 1):
            cell = ws.cell(row=ri, column=ci, value=row.get(col,''))
            cell.font = ROW_FONT
            if ri % 2 == 0: cell.fill = ALT_FILL

    # Auto-width (capped at 40)
    for ci, col in enumerate(cols, 1):
        max_w = max(len(str(col)), max((len(str(r.get(col,''))) for r in rows), default=0))
        ws.column_dimensions[get_column_letter(ci)].width = min(max_w + 3, 40)

    ws.freeze_panes = 'A2'

wb = Workbook()

# Sheet 0: README
ws0 = wb.active; ws0.title = 'README'
_readme = [
    ('Sheet', 'Contents'),
    ('overall_metrics',   'Overall detection P/R/F1 per pipeline (+ FN breakdown, classification accuracy)'),
    ('per_class',         'P/R/F1 per pipeline x class'),
    ('confusion_matrix',  'GT class vs predicted class counts (incl. bg_rejected and missed)'),
    ('threshold',         'Precision/Recall/F1 at every confidence threshold per pipeline'),
    ('per_plot_det',      'Class-agnostic detection metrics per camera folder per pipeline'),
    ('per_plot_cls',      'Per-class metrics per camera folder per pipeline'),
    ('combined',          'Crop + YOLO union recall analysis per class'),
    ('unique_detections', 'GT insects only detected by each pipeline exclusively'),
    ('raw_detections',    'Every bbox row used in evaluation (tp/fp/fn_*)'),
]
_write_sheet(ws0, [{'Sheet':s,'Contents':c} for s,c in _readme[1:]], ['Sheet','Contents'])
ws0['A1'].value = 'Sheet'; ws0['B1'].value = 'Contents'  # already set by _write_sheet

# Sheets 1-9
_sheets = [
    ('overall_metrics',   _xls_overall),
    ('per_class',         _xls_per_class),
    ('confusion_matrix',  _xls_confusion),
    ('threshold',         _xls_threshold),
    ('per_plot_det',      _xls_plot_det),
    ('per_plot_cls',      _xls_plot_cls),
    ('combined',          _xls_combined),
    ('unique_detections', _xls_unique),
    ('raw_detections',    _xls_raw),
]
for name, data in _sheets:
    _ws = wb.create_sheet(name)
    _write_sheet(_ws, data)

_xlsx_path = EVAL_DIR / f'eval_{RUN_TS}.xlsx'
wb.save(str(_xlsx_path))

# ── Write captured print output to report.txt ────────────────────────
_report = {'run_timestamp': RUN_TS, 'sections': _report_sections}
(EVAL_DIR / 'report.json').write_text(_json.dumps(_report, indent=2, ensure_ascii=False))

print(f'\n\u2713 Saved to {EVAL_DIR}/')
print(f'   └ eval_{RUN_TS}.xlsx   ← all metrics (9 sheets)')
print(f'   └ report.json         ← text output split by section')
print(f'   └ threshold_analysis.png')
print(f'   Sheets: {[s for s,_ in _sheets]}')
